In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString
import os


In [ ]:
import pickle
data_path = "../../data/hcm_data/"

In [ ]:
with open(os.path.join(data_path,"nwk_hcm/hcm_edges_poi_new_simplify.pkl"), 'rb') as f:
    edgeinfo = pickle.load(f)

with open(os.path.join(data_path,"nwk_hcm/hcm_nodes_new.pkl"), 'rb') as f:
    nodeinfo = pickle.load(f)

In [ ]:
import networkx as nx  # or compute manually

# edgeinfo: dict of {idx: [highway, length, node_u, node_v, poi*9, ...]}

# Build degree maps by scanning all edges
in_degree = {}   # node -> count of edges where node_v == node
out_degree = {}  # node -> count of edges where node_u == node

for idx, edge in edgeinfo.items():
    node_u = edge[2]
    node_v = edge[3]
    out_degree[node_u] = out_degree.get(node_u, 0) + 1
    in_degree[node_v]  = in_degree.get(node_v, 0) + 1

# Inject into edgeinfo
for idx, edge in edgeinfo.items():
    node_u = edge[2]
    node_v = edge[3]
    deg_out = out_degree.get(node_u, 0)  # how many edges leave node_u
    deg_in  = in_degree.get(node_v, 0)   # how many edges arrive at node_v
    edge.append(deg_out)
    edge.append(deg_in)

In [ ]:
import numpy as np

deg_in_vals  = [edge[13] for edge in edgeinfo.values()]
deg_out_vals = [edge[14] for edge in edgeinfo.values()]

for name, vals in [("deg_in", deg_in_vals), ("deg_out", deg_out_vals)]:
    arr = np.array(vals)
    print(f"{name}: min={arr.min()}, max={arr.max()}, mean={arr.mean():.2f}, "
          f"median={np.median(arr):.1f}, std={arr.std():.2f}, "
          f"p95={np.percentile(arr,95):.1f}, p99={np.percentile(arr,99):.1f}")

In [ ]:
sample_edge = edgeinfo[2]
sample_u = sample_edge[2]
sample_v = sample_edge[3]

In [ ]:
unique_edges = set()

for edge in edgeinfo.values():
    u = edge[2]
    v = edge[3]
    unique_edges.add((min(u, v), max(u, v)))

In [ ]:
total_edges = len(edgeinfo)
unique_undirected = len(unique_edges)

print("total_edges:", total_edges)
print("unique_undirected:", unique_undirected)
print("ratio:", total_edges / unique_undirected)

In [ ]:
deg = {}

for edge in unique_edges:
    u = edge[0]
    v = edge[1]

    deg[u] = deg.get(u, 0) + 1
    deg[v] = deg.get(v, 0) + 1

In [ ]:
edge_deg_start = []
edge_deg_end = []

for edge in edgeinfo.values():
    u = edge[2]
    v = edge[3]

    edge_deg_start.append(deg[u])
    edge_deg_end.append(deg[v])

In [ ]:
import numpy as np

deg_in_vals = np.array(edge_deg_start)
deg_out_vals = np.array(edge_deg_end)

def print_stats(name, arr):
    print(f"\n{name} stats:")
    print(f"  count: {len(arr)}")
    print(f"  mean: {arr.mean():.4f}")
    print(f"  std: {arr.std():.4f}")
    print(f"  min: {arr.min()}")
    print(f"  max: {arr.max()}")
    print(f"  median: {np.median(arr)}")

print_stats("deg_in", deg_in_vals)
print_stats("deg_out", deg_out_vals)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.hist(deg_in_vals, bins=50)
plt.title("In-degree distribution")
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.show()

plt.figure()
plt.hist(deg_out_vals, bins=50)
plt.title("Out-degree distribution")
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure()
plt.hist(deg_in_vals, bins=50)
plt.yscale("log")
plt.title("In-degree distribution (log scale)")
plt.show()